# Lecture 6 — How many groups, and how would you know?

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opherdonchin/AdvancedStatisticsCourse_36725561/blob/main/notebooks/lecture6_penguin_groups.ipynb)


In the lecture, penguins of two species were fitted by k-means and by EM for a Gaussian mixture, with the number of groups $K$ given as 2. Here the file holds **all 342 penguins** (the lecture's file was a subset), four measurements each (bill length, bill depth, flipper length, body mass), with **no labels and no given $K$**. All four are used throughout.

**How many groups does this population have, and what could "good" mean when the groups are hidden?**

We take "good" to mean a low held-out log score: how well a fitted model describes penguins it was not fitted to. That is a statement about the density of the four measurements on new penguins; it says nothing about species or about any other task.

| Section | What we do, and what it contributes |
|---|---|
| 0. Data and design | Load the file; hold out penguins, which is how every score is made |
| 1. The E-step | Write the lecture's E-step and check it against scikit-learn |
| 2. EM from a k-means++ start | Run the lecture's EM and check it against scikit-learn |
| 3. k-means, held out | Why distortion cannot choose $K$ |
| 4. The mixture, held out | Held-out log score for two covariance families: a score that can choose $K$, which the training score cannot; why the family changes the answer; the fit checked by drawing penguins from it |
| The answer | The opening question answered from Sections 3 and 4 |

There are 9 questions: six code and three written.

## 0. Data and design

### *Explanation to read*

#### The data, the standardization and the held-out design

The data are $x_n\in\mathbb R^4$ for $n=1,\dots,342$: bill length (mm), bill depth (mm), flipper length (mm) and body mass (g) of Palmer penguins (Horst, Hill and Gorman, 2020; CC0). No group is recorded, and nothing in the file says how many there are.

The measurements have different units, so we **standardize** each one: subtract a mean and divide by a standard deviation. This is a modeling choice, since it changes what k-means calls close. Whenever a score is held out, the mean and standard deviation come from the *training* penguins only, so nothing about the held-out penguins leaks into the fit.

To score a choice of $K$ on penguins the fit has not seen, we use **5-fold cross-validation**: split the penguins at random into five folds; fit on four (273 or 274 penguins) and score on the fifth (68 or 69); rotate so each fold is held out once, and average; every penguin is held out exactly once. A mixture is scored on its held-out penguins by L04's log score, in nats per penguin; lower is better. This notebook uses one such split, and reports four repeated splits (20 folds) where one split can mislead.

**Library calls.** `GaussianMixture(K, init_params="k-means++", n_init=1).fit(A)` runs EM from one k-means++ start. For a fitted `g`: `g.score(B)` is the mean log-likelihood per row of `B`, so L04's log score of those rows is `-g.score(B)`, and `-g.score_samples(B)` is the log score of each row separately; `g.predict_proba(B)` the responsibilities, and `g.weights_`, `g.means_`, `g.covariances_` hold $\pi_k$, $\mu_k$, $\Sigma_k$. `KMeans(K, init="k-means++", n_init=1).fit(A)` fits k-means from one k-means++ start. The start is random; we fix `random_state`, so every run gives the same numbers. Documentation: [`GaussianMixture`](https://scikit-learn.org/stable/modules/generated/sklearn.mixture.GaussianMixture.html), [`KMeans`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html).

### *Code to run*

#### Load the data and make the folds

Loads the penguin file from the course data repository and makes the 5 folds (seed 2027). `standardized(train, test)` standardizes both parts with the training mean and standard deviation. `Z` is *all* penguins standardized once, for the work where nothing is held out. The **pair plot** shows all four measurements at once: each panel off the diagonal is a scatter of one pair, each panel on the diagonal a histogram of one measurement.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "2")      # before NumPy loads: avoids slow thread oversubscription in the sweeps
import time
from urllib.request import urlopen
import warnings
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy.stats import multivariate_normal as mvn
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import RepeatedKFold

warnings.filterwarnings("ignore")      # scikit-learn convergence messages in the sweeps
np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.dpi"] = 110

data_url = (
    "https://raw.githubusercontent.com/opherdonchin/"
    "AdvancedStatisticsCourse_36725561/main/data/penguins_all_species.csv"
)
with urlopen(data_url, timeout=30) as data_file:
    X = np.loadtxt(data_file, delimiter=",", skiprows=1)
N, D = X.shape
assert (N, D) == (342, 4) and np.all(np.isfinite(X))
print(f"{N} penguins, {D} measurements")
print("means:", X.mean(0), " standard deviations:", X.std(0))

N_REPEATS = 1     # one 5-fold split; 4 repeats make 20 folds
folds = list(RepeatedKFold(n_splits=5, n_repeats=N_REPEATS, random_state=2027).split(X))


def standardized(train_idx, test_idx):
    mean, sd = X[train_idx].mean(0), X[train_idx].std(0)        # training penguins only
    return (X[train_idx] - mean) / sd, (X[test_idx] - mean) / sd


Z = (X - X.mean(0)) / X.std(0)

names = ["bill length (mm)", "bill depth (mm)", "flipper length (mm)", "body mass (g)"]
penguins = pd.DataFrame(X, columns=names)
sns.pairplot(penguins, height=1.7, plot_kws=dict(s=10, alpha=0.6, color="0.3"), diag_kws=dict(color="0.5"))
plt.show()

#### Section summary

The 342 penguins are loaded and standardized with training-penguin statistics, and five folds fix which penguins are held out in each score.

## 1. The E-step

### *Explanation to read*

#### The mixture and its responsibilities

The model is the lecture's mixture: each penguin has a hidden group $z_n\in\{1,\dots,K\}$ with $p(z_n=k)=\pi_k$, and given the group the four measurements are Gaussian, $p(x_n\mid z_n=k)=\mathcal N(x_n\mid\mu_k,\Sigma_k)$. With $\theta=\{\pi_k,\mu_k,\Sigma_k\}$ the density of a penguin and the log-likelihood of the file are

$$
p(x_n\mid\theta)=\sum_{k=1}^K\pi_k\,\mathcal N(x_n\mid\mu_k,\Sigma_k),\qquad \ell(\theta)=\sum_{n=1}^N\ln p(x_n\mid\theta).\tag{1}
$$

The E-step computes each penguin's **responsibilities**, the posterior probability of its group given $\theta$:

$$
r_{nk}=p(z_n=k\mid x_n,\theta)=\frac{\pi_k\,\mathcal N(x_n\mid\mu_k,\Sigma_k)}{\sum_{j=1}^K\pi_j\,\mathcal N(x_n\mid\mu_j,\Sigma_j)}.\tag{2}
$$

The denominator of (2) is $p(x_n\mid\theta)$ from (1), so one pass over the penguins gives both $r_{nk}$ and $\ell(\theta)$.

### *Code to run*

#### A fitted three-group mixture to practise on

Fits a 3-group mixture with full covariances (each group has its own $4\times4$ matrix) to all 342 penguins, from one k-means++ start. $K=3$ is taken only to have parameters $\theta$ to work with; whether it is right is the notebook's question. The pair plot colors each penguin by its most responsible group, $\arg\max_k r_{nk}$.

In [ ]:
gm = GaussianMixture(3, covariance_type="full", init_params="k-means++", n_init=1, random_state=0).fit(Z)
print("weights:", gm.weights_)
print("shapes:", gm.weights_.shape, gm.means_.shape, gm.covariances_.shape)

penguins["group"] = gm.predict(Z) + 1          # the most responsible group of each penguin
sns.pairplot(penguins, hue="group", palette="deep", height=1.7, plot_kws=dict(s=10, alpha=0.7))
plt.show()

### *Questions to answer*

#### 1. The E-step and the log-likelihood — code

Write `e_step(A, w, mu, S)`. `A` is an $n\times4$ array of standardized penguins; `w`, `mu`, `S` hold $\pi_k$, $\mu_k$, $\Sigma_k$ with the shapes just printed. Return `r`, the $n\times K$ responsibilities (2), and `ell`, the log-likelihood (1) of the rows of `A`. `mvn(mu[k], S[k]).pdf(A)` gives $\mathcal N(x_n\mid\mu_k,\Sigma_k)$ for every row. `np.column_stack` puts the $K$ resulting columns side by side into an $n\times K$ array, and `weighted / density[:, None]` divides row $n$ by `density[n]`. The next cell tests your function.

In [ ]:
# answer here

### *Code to run*

#### Check the E-step against the library

Compares your results with `gm.predict_proba` and `gm.score` (times $N$); the largest difference should be at rounding-error size, about $10^{-15}$. It also counts the penguins whose largest responsibility is below 0.9 and prints the least certain one.

In [ ]:
r_hand, ell_hand = e_step(Z, gm.weights_, gm.means_, gm.covariances_)
print("largest difference in responsibilities:", np.abs(r_hand - gm.predict_proba(Z)).max())
print(f"log-likelihood: ours {ell_hand:.4f}, library {gm.score(Z) * N:.4f}")
sure = r_hand.max(1)
print("penguins with no responsibility above 0.9:", int((sure < 0.9).sum()), "of", N)
print("least certain penguin", sure.argmin(), "has responsibilities", r_hand[sure.argmin()])

#### Section summary

The hand-written E-step reproduces the library's responsibilities and log-likelihood to rounding error.

## 2. EM and its start

### *Explanation to read*

#### The whole algorithm, and its start

EM alternates the E-step with the lecture's M-step: with $N_k=\sum_nr_{nk}$, the new parameters are the responsibility-weighted averages

$$
\pi_k=\frac{N_k}{N},\qquad \mu_k=\frac1{N_k}\sum_nr_{nk}x_n,\qquad \Sigma_k=\frac1{N_k}\sum_nr_{nk}(x_n-\mu_k)(x_n-\mu_k)^\top.\tag{3}
$$

The code below supplies (3) as `m_step`, with a tiny ridge $10^{-6}I$ added to each $\Sigma_k$ (the library's default `reg_covar`); the ridge only keeps the matrices invertible. The start is the lecture's: k-means++ centers, hard assignments, and one M-step. The k-means++ draw is random; `random_state` fixes it. The code runs 100 rounds of E-step (your `e_step`) and M-step, recording $\ell$, and compares with the library from the same start. From here on every fit, k-means or mixture, uses one k-means++ start.

### *Code to run*

#### EM from a k-means++ start, and the library from the same start

Prints $\ell$ at several iterations and the smallest one-step change (zero up to rounding error), compares the final $\ell$ and means with the library's, and plots $\ell$.

In [ ]:
ridge = 1e-6


def m_step(A, r):
    Nk = r.sum(0)
    w, mu = Nk / len(A), (r.T @ A) / Nk[:, None]
    S = np.array([(r[:, k, None] * (A - mu[k])).T @ (A - mu[k]) / Nk[k] + ridge * np.eye(A.shape[1])
                  for k in range(len(Nk))])
    return w, mu, S


K = 3
labels0 = KMeans(K, init="k-means++", n_init=1, random_state=1).fit(Z).labels_
w, mu, S = m_step(Z, np.eye(K)[labels0])           # the start: one M-step from k-means++ hard assignments
start = (w.copy(), mu.copy(), S.copy())
ell = []
for it in range(100):
    r, l = e_step(Z, w, mu, S)
    ell.append(l)
    w, mu, S = m_step(Z, r)
ell = np.array(ell)
print("ell at iterations 0, 1, 2, 5, 10, 20, 99:", ell[[0, 1, 2, 5, 10, 20, 99]].round(2))
print("smallest one-step change in ell:", np.diff(ell).min())

lib = GaussianMixture(K, covariance_type="full", weights_init=start[0], means_init=start[1],
                      precisions_init=np.linalg.inv(start[2]), max_iter=100, tol=0).fit(Z)
print(f"final ell: ours {e_step(Z, w, mu, S)[1]:.6f}   library {lib.score(Z) * N:.6f}")
print("largest difference in means:", np.abs(mu - lib.means_).max())

fig, ax = plt.subplots(figsize=(4.8, 3.2))
ax.plot(ell, marker="o", ms=3)
ax.set(xlabel="EM iteration", ylabel=r"$\ell(\theta)$", title="Log-likelihood by iteration")
plt.show()

#### Section summary

Each round of E-step and M-step leaves $\ell$ the same or larger, and the library from the same start ends where our loop does.

## 3. k-means, held out

### *Explanation to read*

#### Distortion as a score, and the sweep that makes every table below

The lecture's k-means objective is the **distortion**: the mean over penguins of the squared distance to the nearest center. Can it choose $K$? For each fold we fit k-means for $K=1,\dots,12$ (one k-means++ start) on the standardized training penguins and compute the distortion on the training and on the held-out penguins.

The same loop also fits a Gaussian mixture for each $K$, with **full** and with **diagonal** covariances ($\Sigma_k$ a full matrix, or a diagonal one), and keeps its log score (L04's, lower is better) on the training penguins and on each held-out penguin.

### *Code to run*

#### The sweep over $K$, and k-means distortion

Takes a few seconds. Prints the mean k-means distortion per penguin for each $K$, plots it with its fold-to-fold standard deviation, and counts the folds in which held-out distortion rises somewhere between consecutive $K$. The sweep also stores the mixtures' log scores: `sweep[cov]["train"]` and `sweep[cov]["test"]` hold the mean training and held-out score of each fold, by fold and $K$, and `penguin_score[cov]` holds the held-out score of each penguin, by penguin and $K$ (342 rows; the average over repeats when there are several).

In [ ]:
Ks = list(range(1, 13))
t0 = time.time()
km_train, km_test = np.zeros((len(folds), 12)), np.zeros((len(folds), 12))
sweep = {c: {key: np.zeros((len(folds), 12)) for key in ("train", "test")} for c in ("full", "diag")}
penguin_score = {c: np.zeros((N, 12)) for c in ("full", "diag")}      # held-out log score of each penguin, by K


for f, (tr, te) in enumerate(folds):
    A, B = standardized(tr, te)
    for K in Ks:
        km = KMeans(K, init="k-means++", n_init=1, random_state=0).fit(A)
        dist = lambda P: ((P[:, None, :] - km.cluster_centers_[None]) ** 2).sum(2).min(1).mean()
        km_train[f, K - 1], km_test[f, K - 1] = dist(A), dist(B)
        for cov in ("full", "diag"):
            g = GaussianMixture(K, covariance_type=cov, init_params="k-means++", n_init=1, random_state=0).fit(A)
            s = sweep[cov]
            s["train"][f, K - 1], s["test"][f, K - 1] = -g.score(A), -g.score(B)       # log scores: minus the mean log-likelihood
            penguin_score[cov][te, K - 1] += -g.score_samples(B) / N_REPEATS            # each penguin's own held-out log score
print(f"{len(folds)} folds done in {time.time() - t0:.0f} s")


def show(name, M, digits=3):
    print(f"{name:<34}", " ".join(f"{v:7.{digits}f}" for v in M.mean(0)))


print(f"{'K':<34}", " ".join(f"{k:7d}" for k in Ks))
show("k-means distortion, training", km_train)
show("k-means distortion, held out", km_test)

fig, ax = plt.subplots(figsize=(5, 3.4))
ax.errorbar(Ks, km_train.mean(0), km_train.std(0, ddof=1), marker="o", ms=3, label="training")
ax.errorbar(Ks, km_test.mean(0), km_test.std(0, ddof=1), marker="o", ms=3, label="held out")
ax.set(xlabel="number of groups K", ylabel="distortion per penguin", title="k-means distortion against K")
ax.legend()
plt.show()
rises = np.diff(km_test, axis=1) > 0
print("folds in which held-out distortion rises between some K and K+1:", int(rises.any(1).sum()), "of", len(folds))

### *Questions to answer*

#### 2. Why distortion cannot choose $K$ — written answer

The table and plot above show the held-out distortion falling with every added group, although the held-out penguins played no part in placing the centers. k-means describes the data only by its $K$ centers (the lecture's comparison of k-means with EM: no covariances, no weights) and scores a penguin only by its squared distance to the nearest center. Explain why a held-out penguin tends to end up nearer to a center as $K$ grows.

- answer here

#### Section summary

Held-out distortion fell with every added group, so it cannot choose $K$: k-means has no density to score a new penguin with.

## 4. The mixture, held out

### *Explanation to read*

#### The held-out log score and two covariance families

A mixture is scored as in L04. For a held-out penguin $m$ the **log score** is $s_m=-\ln p(x_m\mid\hat\theta)$, with $p$ the mixture density (1) and $\hat\theta$ fitted on the training penguins (L04 wrote this per-case score $\ell_m$; here $\ell$ is the log-likelihood). Its mean over $M$ held-out penguins is

$$
\hat S=\frac1M\sum_{m=1}^M-\ln p(x_m\mid\hat\theta),\tag{4}
$$

in nats per penguin; lower is better, as in L04, and only differences matter. For any set of $M$ penguins, $\hat S=-\ell(\hat\theta)/M$, with $\ell$ the log-likelihood (1) of that set. So the $\ell$ that EM raised in Section 2 is the negative of this score: raising $\ell$ lowers the log score of the training penguins, which is the same mean over the penguins the fit was made on. In one 5-fold split every penguin is held out once, so $M=342$, and each $s_m$ comes from the model fitted without penguin $m$.

We fit two **covariance families** for each $K$. *Full*: each $\Sigma_k$ is a $4\times4$ matrix with 10 free numbers, so a group's ellipse can tilt. *Diagonal*: each $\Sigma_k$ is diagonal, with 4 free numbers, so the ellipse is aligned with the measurement axes and the measurements are uncorrelated within a group. The family is part of the model, so the answer to "how many groups" may depend on it. A $K$-group model has $p=(K-1)+14K$ free numbers with full covariances (the $K-1$ free weights and, per group, 4 means and 10 covariance entries) and $p=(K-1)+8K$ with diagonal ones.

### *Questions to answer*

#### 3. The held-out score of one $K$ — code

For the first fold (`tr, te = folds[0]`, `A, B = standardized(tr, te)`), fit $K=3$ with full covariances to `A`, as for `gm` but with `random_state=0`, and print its log score on `A` and on `B` (minus `g.score`), beside `sweep["full"]["train"][0, 2]` and `sweep["full"]["test"][0, 2]`. (Check: the pairs are equal.)

In [ ]:
# answer here

### *Code to run*

#### Held-out and training log score against $K$

Prints each family's held-out and training log score per penguin (held out: the mean of the 342 penguins' scores; training: the mean over folds). The left plot shows the held-out score of both families, with the standard error $\mathrm{sd}(s)/\sqrt M$ of the 342 scores as bars; the right plot shows the training and the held-out score for full covariances. It also prints, for each family, the $K$ with the lowest held-out score, its number of free numbers $p$ (the counts in the explanation above) and that score.

In [ ]:
print(f"{'K':<34}", " ".join(f"{k:7d}" for k in Ks))
for cov in ("full", "diag"):
    show(f"{cov}: held-out log score / penguin", penguin_score[cov], 2)
    show(f"{cov}: training log score / penguin", sweep[cov]["train"], 2)

def n_params(K, cov):
    return (K - 1) + (14 if cov == "full" else 8) * K          # free numbers: weights, means, covariance entries


for cov in ("full", "diag"):
    K = penguin_score[cov].mean(0).argmin() + 1
    print(f"{cov}: best K {K}, {n_params(K, cov)} free numbers, held-out log score {penguin_score[cov].mean(0).min():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), sharex=True)
for cov, color in (("full", "C0"), ("diag", "C1")):
    s = penguin_score[cov]
    axes[0].errorbar(Ks, s.mean(0), s.std(0, ddof=1) / np.sqrt(N), color=color, marker="o", ms=3, label=cov)
axes[0].set(xlabel="K", ylabel="held-out log score per penguin (nats)", title="Held out (mean; bars: standard error)")
axes[0].set_ylim(3, 5)
axes[1].plot(Ks, sweep["full"]["train"].mean(0), marker="o", ms=3, label="training")
axes[1].plot(Ks, penguin_score["full"].mean(0), marker="o", ms=3, label="held out")
axes[1].set(xlabel="K (full covariances)", ylabel="log score per penguin (nats)", title="Training and held out, full covariances")
axes[0].legend(); axes[1].legend()
plt.show()

### *Questions to answer*

#### 4. Why not choose $K$ by the training score? — written answer

With full covariances the training log score falls with every added group, while the held-out score is lowest at $K=3$ and higher beyond (table and right-hand plot above). Why can the training score not choose $K$? (A $K$-group mixture can be seen as a $(K+1)$-group mixture with one group's weight set to 0; a full group has 14 free numbers.)

- answer here

### *Explanation to read*

#### Comparing $K=3$ with a neighbor, penguin by penguin

With full covariances the mean held-out score is lowest at $K=3$, and the curve is flat near its bottom. Is $K=3$ better than a neighbor $K'$ by more than noise? As L04 scored two forecasts on the same flowers, we score two values of $K$ on the same penguins and compare them penguin by penguin. Each penguin is held out once in this split, and `penguin_score["full"][:, K - 1]` holds its held-out log score $s_m(K)$. With $M=342$ penguins, the difference, its mean and its standard error are

$$
d_m=s_m(K')-s_m(3),\qquad \bar d=\frac1M\sum_{m=1}^M d_m,\qquad \mathrm{se}=\frac{\mathrm{sd}(d)}{\sqrt M},\tag{5}
$$

with $\mathrm{sd}(d)$ the sample standard deviation of the $d_m$ (`ddof=1`). A penguin has $d_m>0$ when $K=3$ scored lower (better) on it, and $\bar d=\hat S(K')-\hat S(3)$ is the difference of the two means in (4). L04's rough guide applies: a $\bar d$ more than about two standard errors from 0 is unlikely to come only from which penguins happened to be held out. One difference from L04: there one fixed forecast scored all 75 flowers, while here the held-out scores come from five fitted models, one per fold, so the standard error is a rough guide.

### *Questions to answer*

#### 5. $K=3$ against $K=4$ — code

For full covariances, compute $d_m$ for $K'=4$ from `penguin_score["full"]` (row $m$, column $K-1$). Print $\bar d$, its standard error (5), the ratio $\bar d/\mathrm{se}$ and the number of penguins on which $K=3$ scores lower. (Check: $\bar d$ equals the difference of the $K=4$ and $K=3$ entries of the full held-out row above.)

In [ ]:
# answer here

### *Code to run*

#### The other neighbors, and the best $K$ by fold

Prints, for both families, the number of folds whose best $K$ is 1, 2, ..., 12; then, for full covariances, the comparison of $K=3$ with $K'=2$ and $K'=5$ in the format of Question 5.

In [ ]:
for cov in ("full", "diag"):
    best = np.bincount(sweep[cov]["test"].argmin(1) + 1, minlength=13)[1:]
    print(f"{cov}: number of folds whose best K is 1, 2, ..., 12: {best}")

s = penguin_score["full"]
for other in (2, 5):
    d = s[:, other - 1] - s[:, 2]
    se = d.std(ddof=1) / np.sqrt(len(d))
    print(f"K={other} minus K=3: mean {d.mean():+.3f}, standard error {se:.3f}, ratio {d.mean() / se:.1f}, "
          f"K=3 lower on {(d > 0).sum()} of {len(d)} penguins")

### *Explanation to read*

#### The same comparisons over 20 folds

Setting `N_REPEATS = 4` in the data cell and rerunning from there makes 20 folds in under a minute. Each penguin is then held out four times, once per repeat, so it has four scores for each $K$; averaging them gives one $s_m(K)$ per penguin, and (5) applies with $M=342$ as before. The author's run (seed 2027, `random_state=0`) gave these paired differences for full covariances; the ratio is $\bar d/\mathrm{se}$ computed from unrounded values:

| $K'$ minus $K=3$ | mean $\bar d$ | standard error | ratio | penguins with $K=3$ lower |
|---|---|---|---|---|
| $K'=2$ | $+0.092$ | $0.034$ | $2.74$ | 236 of 342 |
| $K'=4$ | $+0.008$ | $0.021$ | $0.37$ | 134 of 342 |
| $K'=5$ | $+0.058$ | $0.034$ | $1.70$ | 159 of 342 |

The best $K$ by fold, with full covariances, was 2 in 4 folds, 3 in 7, 4 in 4, 5 in 3, and 7 and 8 in one each; with diagonal covariances it ranged from 5 to 11, no value in more than 5 folds.

### *Questions to answer*

#### 6. How firmly is $K=3$ chosen? — written answer

Using your $K=4$ result, the printed $K=2$ and $K=5$ comparisons for this split, and the 20-fold table: for which neighbors does $K=3$ score lower by more than about two standard errors, and for which is it not settled? What would you claim about the number of groups with full covariances?

- answer here

### *Explanation to read*

#### Why a covariance family can change the answer

A diagonal covariance gives an axis-aligned ellipse. A group whose measurements are correlated is a tilted ellipse, which a diagonal model can cover only with several small axis-aligned groups. So where measurements are correlated within groups, the diagonal family needs more groups for the same density, and its held-out log score falls slowly and flattens: best $K$ is 9 on this split, and from 5 to 11 by fold over 20 folds.

Why would measurements be correlated within a group? A penguin's measurements depend on its group, and also on things the file does not record, for example the penguin's sex and its size (which age and condition also move). Each of these is a parent of all four measurements, as the graph drawn below shows: a dashed circle is not measured, and the group, which the mixture includes as $z_n$, is one of them. This is the **fork** of L03: variables that share a parent are dependent until the parent is observed. (L03's other structures, the pipe, the collider and a collider's descendant, do not occur here.) Conditioning on a group, $z_n=k$, does not observe sex or size, so the four measurements remain dependent given the group. This is our reading of why they are correlated, not something the notebook tests.

A full covariance $\Sigma_k$ has off-diagonal entries that represent that dependence as correlations. A diagonal $\Sigma_k$ sets them to 0, which assumes the four measurements independent given the group, as if sex and size did not matter; the diagonal family pays for that assumption with more groups.

### *Code to run*

#### The graph of a penguin's measurements

Draws the graph just described: an arrow runs from each parent to each of the four measurements. Filled circles are measured; dashed circles are not.

In [ ]:
from matplotlib.patches import Circle, FancyArrowPatch

fig, ax = plt.subplots(figsize=(6.6, 3.2))
r = 0.34
parents = [("group $z_n$", 1.5, "C0"), ("sex", 3.0, "0.45"), ("size", 4.5, "0.45")]
children = [("bill\nlength", 0.75), ("bill\ndepth", 2.25), ("flipper\nlength", 3.75), ("body\nmass", 5.25)]
top = [Circle((x, 2.0), r, fill=False, ls="--", ec=color, lw=1.6) for _, x, color in parents]
bottom = [Circle((x, 0.0), r, fc="0.85", ec="0.2", lw=1.2) for _, x in children]
for patch in top + bottom:
    ax.add_patch(patch)
for p in top:
    for c in bottom:
        ax.add_patch(FancyArrowPatch(p.center, c.center, patchA=p, patchB=c, arrowstyle="-|>", mutation_scale=10,
                                     color="0.35", lw=0.9, shrinkA=0, shrinkB=0))
for (label, x, color) in parents:
    ax.text(x, 2.0 + r + 0.12, label, ha="center", va="bottom", color=color)
for label, x in children:
    ax.text(x, -r - 0.12, label, ha="center", va="top")
ax.set(xlim=(-0.3, 6.3), ylim=(-1.2, 2.9), aspect="equal", title="Parents of the four measurements (dashed: not measured)")
ax.axis("off")
plt.show()

### *Questions to answer*

#### 7. The correlation matrix of each group — code

A covariance $\Sigma_k$ mixes the spread of each measurement with how the measurements move together. The **correlation matrix** keeps only the second:
$$
R_k=D_k^{-1/2}\,\Sigma_k\,D_k^{-1/2},\qquad R_{k,ij}=\frac{\Sigma_{k,ij}}{\sqrt{\Sigma_{k,ii}\,\Sigma_{k,jj}}},
$$
where $D_k$ is the diagonal of $\Sigma_k$; every $R_{k,ii}=1$. For each of the three groups of `gm`, compute $R_k$, keep the three in a list `R`, and draw them as heatmaps side by side, with the measurement names on the axes. For one group, `d = np.sqrt(np.diag(S[k]))` gives the standard deviations and `S[k] / np.outer(d, d)` gives $R_k$ (`np.outer(d, d)` is the matrix of products $d_id_j$). `sns.heatmap(R[k], vmin=-1, vmax=1, cmap="RdBu_r", annot=True, fmt=".2f", xticklabels=short, yticklabels=short, ax=axes[k])` draws one panel. (Check: every entry off the diagonal is positive.)

In [ ]:
# answer here

### *Explanation to read*

#### How large is a correlation? An answer in nats

A correlation of 0.6 says that a group's ellipsoid is tilted; it does not say whether the tilt matters for the density. A standard one-number summary of a correlation matrix is its **determinant**: $\det R_k=\det\Sigma_k/\prod_i\Sigma_{k,ii}$ is 1 when the measurements are uncorrelated and falls toward 0 as they become linearly dependent. Its logarithm has a meaning from L04.

Compare the group's Gaussian $\mathcal N(\mu_k,\Sigma_k)$ with the diagonal Gaussian $\mathcal N(\mu_k,D_k)$ that has the same means and the same variances, and average the difference of their log densities over penguins $x$ drawn from the first:
$$
\mathbb E\big[\ln\mathcal N(x;\mu_k,\Sigma_k)-\ln\mathcal N(x;\mu_k,D_k)\big]
=-\tfrac12\ln\det\Sigma_k+\tfrac12\ln\det D_k-\tfrac12\,\mathbb E\big[(x-\mu_k)^\top\Sigma_k^{-1}(x-\mu_k)\big]+\tfrac12\,\mathbb E\big[(x-\mu_k)^\top D_k^{-1}(x-\mu_k)\big]
=-\tfrac12\ln\det R_k .
$$
The two expectations cancel: each equals the number of measurements, 4, because $\mathbb E[(x-\mu_k)(x-\mu_k)^\top]=\Sigma_k$ and both $\Sigma_k^{-1}\Sigma_k$ and $D_k^{-1}\Sigma_k$ have 1 on every diagonal entry. The left side is L04's KL divergence from the full Gaussian to the diagonal one. So $-\tfrac12\ln\det R_k\ge0$ is the amount, in nats per penguin, by which L04's log score of penguins drawn from group $k$'s Gaussian rises (gets worse) when a diagonal covariance replaces the full one, with the means and variances held fixed. For two measurements with correlation $\rho$ it is their mutual information, $-\tfrac12\ln(1-\rho^2)$. L01's conditional standard deviation of one Gaussian measurement given the other is $\sigma_{1|2}=\sigma_1\sqrt{1-\rho^2}$, so $-\tfrac12\ln(1-\rho^2)=\ln(\sigma_1/\sigma_{1|2})$: the mutual information is the logarithm of how many times narrower one measurement's distribution becomes once the other is known (for $\rho=0.7$, the standard deviation falls to $\sqrt{0.51}\approx0.71$ of its former size, which is $0.34$ nats; an illustration of the formula, not a penguin value). It is in the same units, and the same direction (higher is worse), as every held-out log score in this section.

### *Questions to answer*

#### 8. How much the correlations are worth — code

From your list `R`, compute $-\tfrac12\ln\det R_k$ for each group (`np.linalg.det`) and their average weighted by `gm.weights_`. Print them beside two held-out gaps from this section, both from the mean held-out log scores of this split (`penguin_score`): the best diagonal model's score minus the best full model's (`penguin_score[cov].mean(0).min()`), and full $K=4$ minus full $K=3$. In a comment, say whether the within-group correlations are large or small on this scale. (Check: group 1 gives about 1.18 nats.)

In [ ]:
# answer here

### *Explanation to read*

#### Penguins drawn from the fitted model

A fitted mixture should also generate penguins that look like the real ones, and drawing them checks the fit directly. The model's arrows run from the hidden group to the measurements, so a penguin is generated in that order, as L01 drew a state and then a measurement from it: draw the group $z\sim\pi$, then the four measurements $x\sim\mathcal N(\mu_z,\Sigma_z)$. `gm.sample(n)` does this $n$ times and returns two arrays: the draws $x$, of shape $(n,4)$, and the group $z$ of each draw, numbered 0 to 2. It uses the fit's `random_state`, so the draws are the same on every run.

`gm` is the full $K=3$ mixture of Section 1, the model Section 4 chose, fitted to all 342 penguins after standardizing them with `Z`. Its draws are therefore in standardized units: a standardized value $z=(x-\bar x)/s$ uses the mean $\bar x$ and standard deviation $s$ of each measurement over all penguins (`X.mean(0)` and `X.std(0)`), and $x=s\,z+\bar x$ returns it to mm and g.

### *Questions to answer*

#### 9. A simulated population beside the real one — code

Draw 342 penguins from `gm` with `gm.sample(342)` and convert the draws to the original units. Put the 342 real penguins (`X`) and the 342 simulated ones in one table with the four columns `names` and a column `source` whose values are `"real"` and `"simulated"`: `pd.DataFrame(array, columns=names).assign(source="real")` builds one part and `pd.concat([part1, part2], ignore_index=True)` joins them. Draw `sns.pairplot(table, hue="source", height=1.7, plot_kws=dict(s=10, alpha=0.5))` and print the share of the draws in each group beside `gm.weights_`. In a comment, say where the simulated population differs visibly from the real one. (Check: the draws have shape $(342,4)$ and the groups shape $(342,)$; each group's share is within about 0.05 of its $\pi_k$.)

In [ ]:
# answer here

#### Section summary

With full covariances the held-out log score is lowest at $K=3$ ($3.62$ nats per penguin). Over 20 folds $K=3$ scores lower than $K=2$ by 2.7 standard errors and is not separated from $K=4$ (0.37) or $K=5$ (1.7); the training score falls with every $K$. Within each group the four measurements are positively correlated (Question 7), and the correlations are worth 0.5 to 1.2 nats per penguin against a diagonal covariance (Question 8). With diagonal covariances the best $K$ is 9 on this split and varies by fold, and the best diagonal model has $p=80$ free numbers against $p=44$ for the best full one (printed above) and a higher (worse) log score. Penguins drawn from the full three-group fit overlap the real ones in every panel of the pair plot, differing at the extremes (Question 9).

## The answer

### *Explanation to read*

#### Answer to the opening question

**How many groups does this population have, and what could "good" mean when the groups are hidden?**

Here "good" meant a low held-out log score: how well a fitted mixture describes penguins it was not fitted to. That measures the density of the four measurements on new penguins; it does not measure agreement with species or performance on any other task. By that measure the number of groups depends on the covariance family. With full covariances, the held-out log score is lowest at 3 ($3.62$ nats per penguin), 2 is worse (2.7 standard errors over 20 folds), and 4 and 5 are not separated from 3 (0.37 and 1.7), so at least 3 groups are needed and the data do not choose among 3, 4 and 5. With diagonal covariances the lowest score is at 9 groups on this split and varies from 5 to 11 by fold, because the diagonal family covers the within-group correlations with many axis-aligned groups. Neither k-means distortion nor the training log score can choose $K$: both fall with every added group. Penguins drawn from the three-group full fit overlap the real ones in every panel of the pair plot; the visible difference is a few simulated penguins lighter than any real one (Question 9).

**How many groups, and how would you know?** "The number of groups" is a statement about a model family and a criterion. Here it is the number of Gaussian components a family needs to describe the density of these four measurements on new penguins, not a count of populations.